In [92]:
from pathlib import Path

import pandas as pd
import numpy as np

In [93]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR_PATH = PROJECT_ROOT / "data" / "processed"

In [94]:
df: pd.DataFrame = pd.read_csv(
    RAW_DATA_DIR_PATH / "mental-health-in-tech-2016.csv"
)

In [95]:

# Dropped because it is meant to be a global analysis, not one focused on the US only.
# A global analysis wouldn't need this type of data.
df = df.drop(
    columns=[
        "What US state or territory do you live in?",
        "What US state or territory do you work in?",
    ]
)
general_boolean_map: dict = {
    "1": True,
    "0": False,
    "1.0": True,
    "0.0": False,
    "yes": True,
    "no": False,
    "i don't know": np.nan,
    "maybe": np.nan,
    "n/a": np.nan,
    "i'm not sure": np.nan,
    "i am not sure": np.nan,
    "unsure": np.nan,
    'not eligible for coverage / n/a': np.nan,
    "nan": np.nan,
    "not applicable to me": np.nan
}

difficulty_value_map: dict[str, int] = {
    "very easy": 2,
    "somewhat easy": 1,
    "neither easy nor difficult": 0,
    "somewhat difficult": -1,
    "very difficult": -2
}

diagnosis_value_map: dict = {
    "yes, always": 2,
    "sometimes, if it comes up": 1,
    "not applicable to me": np.nan,
    "no, because it doesn't matter": -1,
    "no, because it would impact me negatively": -2
}

extent_value_map: dict[str, int] = {
    "yes, all of them": 1,
    "yes, they all did": 1,
    "some of them": 0,
    "some did": 0,
    "none of them": -1,
    "none did": -1
}

for column_name in df:
    df[column_name] = (
        df[column_name].astype(str).str.lower().str.strip()
    ).replace(general_boolean_map).replace(extent_value_map)

difficulty_value_column = "If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:"
df[difficulty_value_column] = df[difficulty_value_column].replace(difficulty_value_map)

diagnosis_value_columns: list[str] = [
    "If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?", 
    "If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?"
]

df[diagnosis_value_columns] = df[diagnosis_value_columns].replace(diagnosis_value_map)
for column in df:
    print(df[column].unique())

[False True]
<StringArray>
['26-100', '6-25', nan, 'more than 1000', '100-500', '500-1000', '1-5']
Length: 7, dtype: str
[True nan False]
[nan True False]
[nan False True]
[nan True False]
[False True nan]
[False True nan]
[nan True False]
[2 1 0 nan -2 -1]
[False nan True]
[False nan True]
[nan True False]
[True nan False]
[nan True False]
[False nan True]
[nan True False]
<StringArray>
[nan, 'yes, i know several', 'i know some', 'no, i don't know any']
Length: 4, dtype: str
[nan 1 -1 -2 2]
[nan True False]
[nan 1 -2 2 -1]
[nan False True]
[nan True False]
<StringArray>
[nan, '1-25%', '76-100%', '26-50%', '51-75%']
Length: 5, dtype: str
[True False]
['no, none did' 1 0 nan]
<StringArray>
[      'n/a (not currently aware)',             'i was aware of some',
 'yes, i was aware of all of them',   'no, i only became aware later',
                               nan]
Length: 5, dtype: str
[nan -1 0 1]
[-1 0 nan 1]
[nan 'yes, always' 'sometimes' False]
[0 -1 nan 1]
[-1 0 1 nan]
<StringArray

In [96]:

df.to_csv(
    PROCESSED_DATA_DIR_PATH / "processed.csv", index=False, encoding="utf-8"
)